In [3]:
from tqdm.notebook import tqdm

In [4]:
import re

In [5]:
import pandas as pd
import time

In [6]:
import subprocess

In [7]:
import json
import requests

## Helpers for connecting to GitHub v3 APIs

In [6]:
with open("auth_token.txt", "r") as f:
    TOKEN = f.read()

In [7]:
import requests

url = 'https://api.github.com/search/code?q='
headers = {
    'Authorization': f'Bearer {TOKEN.strip()}',
    'Accept': 'application/vnd.github+json',
    'X-GitHub-Api-Version': '2022-11-28'
}

## Getting top 8000 package list from PyPi

In [8]:
# https://hugovk.github.io/top-pypi-packages/top-pypi-packages-30-days.min.json
with open("top-pypi-packages-30-days.min.json", "r") as f:
    pypi_packages = json.load(f)

In [9]:
df_pypi = pd.DataFrame(pypi_packages['rows'])

In [10]:
df_pypi.head()

,download_count,project
0,895246471,boto3
1,424262451,urllib3
2,362439956,botocore
3,324232670,requests
4,302150693,typing-extensions


In [11]:
df_pypi.sort_values(by='download_count', inplace=True, ascending=False)

In [12]:
df_pypi.head()

,download_count,project
0,895246471,boto3
1,424262451,urllib3
2,362439956,botocore
3,324232670,requests
4,302150693,typing-extensions


## Grab the github repo URL from PyPi page

In [13]:
projects = df_pypi.project.tolist()

In [14]:
PYPI_URL = 'https://pypi.org/project/{package}/'
github_repo_pattern = r'https://github\.com/([A-Za-z0-9-]+/[A-Za-z0-9-]+)'
def get_pypi_page(package):
    r = requests.get(PYPI_URL.format(package=package))
    repo = [p for p in list(set(re.findall(github_repo_pattern, r.text))) if 'pypi' not in p]
    return repo

In [16]:
PYPI_URL = 'https://pypi.org/project/{package}/'
repos = {}
for p in tqdm(projects):
    repos[p] = get_pypi_page(p)

  0%|          | 0/8000 [00:00<?, ?it/s]

In [17]:
repos

{'boto3': ['boto/boto3'],
 'urllib3': ['urllib3/urllib3'],
 'botocore': ['boto/botocore', 'boto/boto3', 'aws/aws-cli'],
 'requests': ['psf/requests'],
 'typing-extensions': ['python/typing'],
 'setuptools': ['psf/black', 'astral-sh/ruff', 'pypa/setuptools'],
 'charset-normalizer': ['jawah/niquests',
  'Ousret/charset',
  'ousret/charset',
  'jawah/wassima',
  'nickspring/charset-normalizer-rs',
  'chardet/chardet',
  'PyYoshi/cChardet'],
 'certifi': ['certifi/python-certifi'],
 's3transfer': ['boto/s3transfer'],
 'wheel': ['pypa/wheel'],
 'packaging': ['pypa/packaging'],
 'pyyaml': ['yaml/pyyaml'],
 'python-dateutil': ['dateutil/dateutil'],
 'idna': ['kjd/idna'],
 'grpcio-status': [],
 'cryptography': ['pyca/cryptography'],
 'pip': ['pypa/pip'],
 'six': ['benjaminp/six'],
 'numpy': ['numpy/numpy'],
 'google-api-core': ['googleapis/python-api-core'],
 'importlib-metadata': ['python/importlib', 'psf/black', 'astral-sh/ruff'],
 'awscli': ['aws/aws-cli'],
 'aiobotocore': ['microsoft/pyrigh

In [18]:
m = []
for k in repos.keys():
    m.append({'name': k, 'repos': repos[k]})

In [19]:
df_repos = pd.DataFrame(m)

In [21]:
df_repos['repo_count'] = df_repos.repos.apply(lambda x: len(x))

In [22]:
def main_repo(row):
    repos = row['repos']
    name = row['name']
    if len(repos) == 1:
        return repos[0]

    fil = [x for x in repos if name in x]
    if len(fil) == 1:
        return fil[0]
    return None

In [23]:
df_repos['main_repo'] = df_repos.apply(main_repo, axis=1)

In [24]:
df_repos.to_csv('pypi_top.csv')

In [25]:
repos = df_repos.main_repo.dropna().tolist()

In [28]:
len(repos)

5670

## Filter the repos which make use of bandit
- This makes use of GitHub v3 APIs to find the text `bandit` in the code

In [ ]:
results = []
for repo in tqdm(repos):
    q = 'bandit+in:file+repo:{repo}'
    r = requests.get(url=url+q.format(repo=repo), headers=headers)
    if r.status_code == 403:
        print(f"Got response: {r.text}\nSleeping")
        time.sleep(65)
        print(f"Woke up")
        r = requests.get(url=url+q.format(repo=repo), headers=headers)

    # print(f"Got response: {r.text}")
    x = json.loads(r.text)
    results.extend(x['items'])

  0%|          | 0/424 [00:00<?, ?it/s]

Got response: {"message":"API rate limit exceeded for user ID 16104456. If you reach out to GitHub Support for help, please include the request ID 63F4:5F0C:574903:B40327:6548210A.","documentation_url":"https://docs.github.com/rest/overview/resources-in-the-rest-api#rate-limiting"}
Sleeping


In [36]:
df = pd.json_normalize(results, sep='_')

In [37]:
df.head()

,name,path,sha,url,git_url,html_url,score,repository_id,repository_node_id,repository_name,...,repository_merges_url,repository_archive_url,repository_downloads_url,repository_issues_url,repository_pulls_url,repository_milestones_url,repository_notifications_url,repository_labels_url,repository_releases_url,repository_deployments_url
0,devcontainer.json,.devcontainer/devcontainer.json,d83f255db449e790abcb7617ce7ad0fe70e9118b,https://api.github.com/repositories/65826230/c...,https://api.github.com/repositories/65826230/g...,https://github.com/taynaud/python-louvain/blob...,1.0,65826230,MDEwOlJlcG9zaXRvcnk2NTgyNjIzMA==,python-louvain,...,https://api.github.com/repos/taynaud/python-lo...,https://api.github.com/repos/taynaud/python-lo...,https://api.github.com/repos/taynaud/python-lo...,https://api.github.com/repos/taynaud/python-lo...,https://api.github.com/repos/taynaud/python-lo...,https://api.github.com/repos/taynaud/python-lo...,https://api.github.com/repos/taynaud/python-lo...,https://api.github.com/repos/taynaud/python-lo...,https://api.github.com/repos/taynaud/python-lo...,https://api.github.com/repos/taynaud/python-lo...
1,CONTRIBUTING.md,CONTRIBUTING.md,e1775ee03d0aad78ee6dc2033781146b7eb8905e,https://api.github.com/repositories/605682586/...,https://api.github.com/repositories/605682586/...,https://github.com/di/id/blob/ba260c01d5021943...,1.0,605682586,R_kgDOJBn7mg,id,...,https://api.github.com/repos/di/id/merges,https://api.github.com/repos/di/id/{archive_fo...,https://api.github.com/repos/di/id/downloads,https://api.github.com/repos/di/id/issues{/num...,https://api.github.com/repos/di/id/pulls{/number},https://api.github.com/repos/di/id/milestones{...,https://api.github.com/repos/di/id/notificatio...,https://api.github.com/repos/di/id/labels{/name},https://api.github.com/repos/di/id/releases{/id},https://api.github.com/repos/di/id/deployments
2,Makefile,Makefile,aa45ecb5f78c6343c1fd607d7d83da0e8af5076b,https://api.github.com/repositories/605682586/...,https://api.github.com/repositories/605682586/...,https://github.com/di/id/blob/ba260c01d5021943...,1.0,605682586,R_kgDOJBn7mg,id,...,https://api.github.com/repos/di/id/merges,https://api.github.com/repos/di/id/{archive_fo...,https://api.github.com/repos/di/id/downloads,https://api.github.com/repos/di/id/issues{/num...,https://api.github.com/repos/di/id/pulls{/number},https://api.github.com/repos/di/id/milestones{...,https://api.github.com/repos/di/id/notificatio...,https://api.github.com/repos/di/id/labels{/name},https://api.github.com/repos/di/id/releases{/id},https://api.github.com/repos/di/id/deployments
3,ambient.py,id/_internal/oidc/ambient.py,eadfb8118efa319cc61e0becfc5d3fb01c488358,https://api.github.com/repositories/605682586/...,https://api.github.com/repositories/605682586/...,https://github.com/di/id/blob/ba260c01d5021943...,1.0,605682586,R_kgDOJBn7mg,id,...,https://api.github.com/repos/di/id/merges,https://api.github.com/repos/di/id/{archive_fo...,https://api.github.com/repos/di/id/downloads,https://api.github.com/repos/di/id/issues{/num...,https://api.github.com/repos/di/id/pulls{/number},https://api.github.com/repos/di/id/milestones{...,https://api.github.com/repos/di/id/notificatio...,https://api.github.com/repos/di/id/labels{/name},https://api.github.com/repos/di/id/releases{/id},https://api.github.com/repos/di/id/deployments
4,pyproject.toml,pyproject.toml,a19eb4f6c3ef1795c80c408924665bab36f2ce09,https://api.github.com/repositories/605682586/...,https://api.github.com/repositories/605682586/...,https://github.com/di/id/blob/ba260c01d5021943...,1.0,605682586,R_kgDOJBn7mg,id,...,https://api.github.com/repos/di/id/merges,https://api.github.com/repos/di/id/{archive_fo...,https://api.github.com/repos/di/id/downloads,https://api.github.com/repos/di/id/issues{/num...,https://api.github.com/repos/di/id/pulls{/number},https://api.github.com/repos/di/id/milestones{...,https://api.github.com/repos/di/id/notificatio...,https://api.github.com/repos/di/id/labels{/na

In [38]:
df.to_csv('v3_filter_bandit_pypi_8k_items.csv')

## Clone the repos

In [8]:
df = pd.read_csv('v3_filter_bandit_pypi_8k_items.csv')

In [9]:
df['repo_url'] = df.apply(lambda x: 'https://github.com/' + x['repository_full_name'], axis=1)

In [10]:
len(df.repository_full_name.unique())

283

In [11]:
ignore_repo_list = [
    'https://github.com/PyCQA/bandit'
]

In [12]:
repo_urls = [r for r in df['repo_url'].unique().tolist() if r not in ignore_repo_list]

In [13]:
len(repo_urls)

282

In [ ]:
for url in repo_urls:
    print(f"Fetching {url}")
    subprocess.call("git clone --depth 1 {}".format(url), cwd='repos_pypi')

## Find all instances of nosec

In [1]:
# Run the following command
# rg "nosec" -n -g "*.py" ./repos_pypi 2>&1 | tee nosec_logs.txt